Task:

Split the raw_data column into multiple columns: emp_id, emp_name, department, salary, state.

For each employee, split emp_name into first name and last name.

Finally, explode the name so that each word in the employee’s name becomes a separate row.


In [0]:
data = [
  (1, "110,Saul GoodMan\tSales|30000|Ohio"),
  (2, "111,Kim Wexler\tLegal|45000|Nevada"),
  (3, "112,Mike Ehrmantraut\tSecurity|40000|Arizona")
]

columns = ["id", "raw_data"]

df = spark.createDataFrame(data, columns)
df.show(truncate=False)

In [0]:
from pyspark.sql import functions as F
df_raw = df.withColumn("raw_split",F.split(F.col("raw_data"),"[,\t|]")) \
            .withColumn("emp_id",F.col("raw_split").getItem(0)) \
            .withColumn("name_part",F.col("raw_split").getItem(1)) \
            .withColumn("name_part",F.split("name_part"," ")) \
            .withColumn("firstname",F.col("name_part").getItem(0)) \
            .withColumn("lastname",F.col("name_part").getItem(1)) \

display(df_raw)

In [0]:
from pyspark.sql.functions import split, col, explode, trim

# Split raw_data into emp_id, emp_name, department, salary, state
df_split = df.withColumn("raw_split", split(col("raw_data"), "[,\t|]")) \
    .withColumn("emp_id", col("raw_split").getItem(0)) \
    .withColumn("emp_name", col("raw_split").getItem(1)) \
    .withColumn("department", col("raw_split").getItem(2)) \
    .withColumn("salary", col("raw_split").getItem(3)) \
    .withColumn("state", col("raw_split").getItem(4)) \
    .drop("raw_split")

display(df_split)

In [0]:
# Split emp_name into first_name and last_name
df_names = df_split.withColumn("name_parts", split(col("emp_name"), " ")) \
    .withColumn("first_name", col("name_parts").getItem(0)) \
    .withColumn("last_name", col("name_parts").getItem(1)) \
    .drop("name_parts")

# Explode emp_name into separate rows for each word
df_exploded = df_names.withColumn("name_word", explode(split(col("emp_name"), " ")))

display(df_exploded)